In [44]:
from models import *
import torch
from dataloaders import *
import torch.nn as nn
from torch.utils.data import DataLoader

In [45]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [46]:
def asymmetry(A, norm='fro', relative=False):
    """
    Compute ||(A - A^T)/2|| as a measure of asymmetry.
    Accepts A of shape (n, n) or (1, n**2) / (n**2,) which is reshaped to square.
    """
    A = A.squeeze()  # drop leading singleton dims, e.g. (1, n**2) -> (n**2,)
    
    if A.dim() == 1:
        n2 = A.shape[0]
        n = int(round(n2 ** 0.5))
        assert n * n == n2, f"Length {n2} is not a perfect square"
        A = A.reshape(n, n)
    
    assert A.dim() == 2 and A.shape[0] == A.shape[1], "A must be square after reshape"
    
    skew = 0.5 * (A - A.T)
    
    if norm == 'fro':
        num = torch.linalg.norm(skew, ord='fro')
        denom = torch.linalg.norm(A, ord='fro')
    elif norm in ('op', 2):
        num = torch.linalg.norm(skew, ord=2)
        denom = torch.linalg.norm(A, ord=2)
    elif norm == 'inf':
        num = torch.linalg.norm(skew, ord=float('inf'))
        denom = torch.linalg.norm(A, ord=float('inf'))
    else:
        raise ValueError(f"Unknown norm: {norm}")
    
    return num / denom.clamp(min=1e-12) if relative else num

In [47]:
tropical_model = SimpleTransformerModel(d_model=64, n_heads=4, num_layers=4, tropical=True, num_classes=64, activation='relu',
                           tropical_attention_cls = TropicalAttention(64, 4, torch.device('cuda')), pool=True, skip=True).to(device)
vanilla_model = SimpleTransformerModel(d_model=64, n_heads=4, num_layers=4, tropical=False, num_classes=64, activation='relu',
                           tropical_attention_cls = None, pool=True, skip=True).to(device)

In [48]:
num_train_samples = 50000
num_val_samples = 10000
n = 8
low_train = 1
high_train = 15
low_test = 1
high_test = 15
use_integer = True
num_additional_node = 0
batch_size = 1
shuffle = True
val_dataset = FloydWarshallDataset(num_val_samples, adversarial_range=(10, 20), length_range=(8, 8), noise_prob=0, value_range=(1, 15))
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [49]:
def check_asymmetry(model, ckpt_path, val_loader, device):
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    result = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            result += asymmetry(out)
    result /= len(val_loader)
    return result

In [50]:
paths = ['15_exp/models/FloydWarshallDataset_tropical_0.0001_20000_20260417_041614_relu_best.pth', #tropical on mix
        '15_exp/models/FloydWarshallDataset_tropical_0.0001_20000_20260416_150701_relu_best.pth', #tropical only on graphs
        '15_exp/models/FloydWarshallDataset_vanilla_0.0001_20000_20260417_031357_relu_best.pth'] #vanilla only on graphs
models = [tropical_model, tropical_model, vanilla_model]

In [51]:
for i in range(len(paths)):
    ckpt_path = paths[i]
    model = models[i]
    asymmetry_curr = check_asymmetry(model, ckpt_path, val_loader, device)
    print(f"Average asymmetry of model {i+1} is {asymmetry_curr}")

Average asymmetry of model 1 is 0.08488713204860687
Average asymmetry of model 2 is 0.004681156016886234
Average asymmetry of model 3 is 0.011281784623861313
